# 📝 과제 02: 날짜로 요일 맞히기

## 만들 결과

달력 규칙(달마다 다른 날수, 윤년)을 하나도 알려 주지 않은 DNN이 `연`, `월`, `일`만 보고 요일을 맞힐 수 있을까요? 같은 날짜를 두 가지로 넣어 비교합니다.

- 숫자 입력: 2024년 10월 20일 -> `2024, 10, 20` (3칸)
- 원-핫 입력: 2024년 10월 20일 -> 연 30칸, 월 12칸, 일 31칸 가운데 `2024`, `10`, `20` 자리만 1 (73칸)

모델 구조와 학습 설정은 같게 두고 입력만 바꿉니다. 결과는 표와 그래프로 확인합니다.

- 숫자 입력의 학습 기록과, 학습률을 바꾼 비교 표
- 원-핫 입력의 학습 기록과, 두 입력의 정확도 곡선
- 옵티마이저와 학습률을 바꾼 학습 손실 곡선
- 2024년까지로 학습하고 2025~2029년을 맞힌 결과 막대그래프

실무 데이터에도 매장 코드, 지역 번호처럼 숫자로 적힌 열이 많습니다. 이런 열을 숫자 그대로 넣을지 원-핫으로 펼칠지가 결과를 얼마나 바꾸는지 이 과제로 확인합니다.

## 준비: 데이터와 제공 코드

- **자료:** `data/과제02/dates.csv`(10,958행). 2000-01-01부터 2029-12-31까지 하루가 한 행이고, 열은 `연`, `월`, `일`, `요일`입니다. `요일`은 0=월, 1=화, 2=수, 3=목, 4=금, 5=토, 6=일입니다. 파이썬 `datetime`으로 만든 달력입니다.
- **제공 코드:** 준비 셀이 아래 이름을 만듭니다. 교안 02에서 만든 `train_one_epoch`, `evaluate`도 넣어 두었습니다.

| 이름 | 내용 |
|---|---|
| `dates` | 달력 표(10,958행) |
| `y_all` | 모든 행의 요일(정수 텐서) |
| `train_index`, `test_index` | 무작위로 섞어 나눈 학습 행 번호 8,766개, 테스트 행 번호 2,192개 |
| `y_train`, `y_test` | 학습 행과 테스트 행의 요일 |

- **작성:** `[작성]` 셀의 `...`를 채우고 바로 아래 `[자가채점]` 셀을 실행하세요.
- **찍기 수준:** 요일은 7개라서, 아무것도 배우지 못한 모델의 정확도는 1/7(약 0.14) 안팎, 손실은 log 7 = 1.95 근처입니다. 결과를 읽을 때 이 두 값과 비교하세요.
- **실행 시간:** CPU에서 과제 전체가 1분 안팎입니다.

In [ ]:
import platform
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import torch

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
# 2000~2029년 달력 10,958일
dates = pd.read_csv(data_dir / "과제02" / "dates.csv")
print("행 수:", len(dates))

# 첫 일주일: 요일이 5(토)에서 시작해 0~6을 한 번 돕니다
first_week = dates.head(7)
display(first_week)

In [ ]:
# 정답: 모든 행의 요일. CrossEntropyLoss는 정수(torch.long) 정답을 받습니다
y_all = torch.tensor(dates["요일"].to_numpy(), dtype=torch.long)

# 행 번호를 무작위로 섞어 80%는 학습, 20%는 테스트에 씁니다(시드 0으로 고정)
generator = torch.Generator().manual_seed(0)
order = torch.randperm(len(dates), generator=generator)
n_train = int(len(dates) * 0.8)
train_index = order[:n_train]
test_index = order[n_train:]
y_train = y_all[train_index]
y_test = y_all[test_index]
print("학습 행:", len(train_index), "| 테스트 행:", len(test_index))

In [ ]:
# 교안 02에서 만든 학습, 평가 함수입니다. 실행만 하고 넘어갑니다.
def train_one_epoch(model, loader, loss_fn, optimizer, scheduler=None):
    """미니배치마다 파라미터를 한 번씩 갱신하고, 그 에폭 미니배치 손실의 평균을 돌려줍니다."""
    total_loss = 0.0
    for X_batch, y_batch in loader:
        # 미니배치를 모델이 있는 계산 장치로 옮깁니다
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        # 지난 스텝의 기울기를 지우고, 순전파 -> 손실 -> 역전파 -> 갱신
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()

        # 스케줄러가 있으면 스텝마다 학습률을 바꿉니다
        if scheduler is not None:
            scheduler.step()

        total_loss += loss.item()

    # len(loader)는 미니배치 수입니다
    return total_loss / len(loader)


def evaluate(model, loader, loss_fn):
    """기울기 없이 예측해 (평균 손실, 정확도)를 돌려줍니다."""
    total_loss = 0.0
    correct = 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)
            logits = model(X_batch)

            # 마지막 미니배치는 장 수가 적으므로, 손실에 장 수를 곱해 더한 뒤 전체 장 수로 나눕니다
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()

    n_samples = len(loader.dataset)
    return total_loss / n_samples, correct / n_samples

## 1. 날짜 숫자를 그대로 넣어 학습하기

날짜에서 가장 먼저 떠오르는 입력은 `연`, `월`, `일` 숫자 세 개입니다. 이 입력으로 교안 02와 같은 DNN을 학습해 비교의 출발점을 만듭니다.

### 1-1. 숫자 입력 만들기

**배경**: `연`(2000~2029)과 `일`(1~31)은 크기가 크게 다릅니다. 교안 01처럼 표준화해서 넣어야 학습률 하나로 세 파라미터가 고르게 갱신됩니다.

**요구사항**
- `연`, `월`, `일` 세 열로 float32 텐서를 만들고, **학습 행(`train_index`)의 열 평균과 표준편차**로 표준화해 **`X_number`** 에 담으세요(10,958행 3열).
- `X_number`에서 학습 행만 고른 **`X_number_train`**, 테스트 행만 고른 **`X_number_test`** 를 만드세요.

**예시**: 첫 행(2000년 1월 1일)은 가장 이른 연도, 1월, 1일이라 세 값이 모두 평균보다 작은 -1.6 안팎이 됩니다.

**확인 기준**: `X_number`는 `(10958, 3)` 모양의 float32 텐서이고, 학습 행만 보면 열마다 평균 0, 표준편차 1입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 평균과 표준편차는 학습 행에서만 구하고, 그 값으로 전체 행을 표준화한다.

세부구현:
1. dates 에서 연, 월, 일 열을 골라 to_numpy 로 바꾸고, torch.tensor 의 dtype 에 torch.float32 를 준다.
2. 1의 텐서를 train_index 로 인덱싱한 뒤 dim=0 으로 mean, std 를 구한다(열마다 하나씩).
3. (전체 텐서 - 평균) / 표준편차
4. X_number 를 train_index, test_index 로 인덱싱한다.
```

</details>

In [ ]:
# [작성] 연, 월, 일 숫자를 표준화한 입력을 만드세요.

# 1. 연, 월, 일 세 열을 float32 텐서로
X_number_raw = ...

# 2. 학습 행의 열 평균과 표준편차로 표준화
train_mean = ...
train_std = ...
X_number = ...

# 3. 학습 입력과 테스트 입력
X_number_train = ...
X_number_test = ...
print("첫 행(2000-01-01):", X_number[0])

In [ ]:
# [자가채점] 입력 모양과 표준화 기준을 확인합니다.
assert isinstance(X_number, torch.Tensor) and X_number.shape == (len(dates), 3), "X_number는 행마다 연, 월, 일 세 값이 든 (10958, 3) 텐서입니다. `...`를 채웠는지 확인하세요."
assert X_number.dtype == torch.float32, "X_number는 float32 텐서로 만드세요. torch.tensor에 dtype=torch.float32를 주면 됩니다."

train_mean_check = X_number[train_index].mean(dim=0)
train_std_check = X_number[train_index].std(dim=0)
assert (train_mean_check.abs() < 1e-3).all(), f"학습 행의 열 평균이 0이 아닙니다({train_mean_check.tolist()}). 평균과 표준편차를 전체 행이나 테스트 행이 아니라 학습 행(train_index)에서 구했는지 보세요."
assert ((train_std_check - 1).abs() < 1e-3).all(), f"학습 행의 열 표준편차가 1이 아닙니다({train_std_check.tolist()}). 학습 행의 표준편차로 나눴는지 보세요."
assert torch.equal(X_number_train, X_number[train_index]), "X_number_train은 X_number에서 train_index 행을 고른 것입니다."
assert torch.equal(X_number_test, X_number[test_index]), "X_number_test는 X_number에서 test_index 행을 고른 것입니다."
print("1-1 확인 완료")

### 1-2. 요일 분류 DNN 학습 함수 만들기

**배경**: 이 과제는 입력과 학습 설정만 바꿔 같은 DNN을 여러 번 학습합니다. 교안 02의 학습 과정을 함수 하나로 묶어 두면 실험마다 바뀌는 값만 넘기면 됩니다.

**요구사항**: 골격의 `...` 여덟 곳을 채워 **`train_weekday_model(X_train, y_train, X_test, y_test, lr, epochs, optimizer_name)`** 을 완성하세요.
- 미니배치: 학습용은 배치 크기 64로 섞고, 테스트용은 배치 크기 64로 섞지 않습니다.
- 모델: `nn.Sequential`로 `입력 열 수 -> 128 -> ReLU -> 128 -> ReLU -> 7`을 만들고 `device`로 옮깁니다. 입력 열 수는 `X_train`의 열 수를 씁니다(숫자 입력이면 3).
- 옵티마이저: `"sgd"`는 채워 두었습니다. `"momentum"`은 `torch.optim.SGD`에 `momentum=0.9`를, 그 밖(`"adam"`)은 `torch.optim.Adam`을 학습률 `lr`로 만듭니다.
- 에폭마다: `train_one_epoch`로 한 에폭 학습하고, `evaluate`를 학습용 로더와 테스트용 로더로 한 번씩 불러 학습 정확도와 테스트 정확도를 구합니다. 에폭별 테스트 정확도는 지켜보기만 하고 설정을 고르는 데 쓰지 않습니다.

**예시**: `train_weekday_model(X_number_train, y_train, X_number_test, y_test, lr=0.001, epochs=2, optimizer_name="adam")`은 (모델, 기록 표)를 돌려주고, 기록 표는 아래 모양입니다(값은 예).

| 에폭 | 학습 손실 | 학습 정확도 | 테스트 정확도 |
|---|---|---|---|
| 1 | 1.95 | 0.14 | 0.13 |
| 2 | 1.95 | 0.15 | 0.12 |

**확인 기준**: 숫자 입력 640행으로 학습해 기록 표의 열과 행 수, 모델의 층 구성을 봅니다. 이어서 `"sgd"`, `"momentum"`, `"adam"`마다 1에폭 학습한 학습 손실과 학습 정확도를, 지문대로 만든 함수로 같은 시드(0)에서 미리 구한 기대값과 비교합니다. 배치 크기 64, 학습용 shuffle=True, 옵티마이저에 lr=lr, 모멘텀은 momentum=0.9 가운데 하나라도 다르면 결과가 달라집니다.

<details><summary>힌트</summary>

```text
접근방법:
- 교안 02의 미니배치 학습 루프를 함수 안에 옮긴다. 바뀌는 값(입력, 학습률, 에폭 수, 옵티마이저 이름)은 인자로 받는다.

세부구현:
1. TensorDataset 으로 입력과 정답을 묶어 DataLoader 에 넘긴다. 학습용만 shuffle 을 켠다.
2. 첫 nn.Linear 의 입력 크기는 X_train.shape 의 둘째 값, 마지막 nn.Linear 의 출력 크기는 7 이다.
3. momentum 과 adam 도 sgd 줄처럼 model.parameters() 와 lr 을 넘긴다.
4. train_one_epoch 의 반환값이 학습 손실이다. evaluate 는 (평균 손실, 정확도) 두 값을 돌려주므로 둘째 값을 쓴다.
```

</details>

In [ ]:
# [작성] 요일 분류 DNN을 학습하는 함수를 완성하세요.
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


def train_weekday_model(X_train, y_train, X_test, y_test, lr, epochs, optimizer_name):
    """요일 분류 DNN을 학습하고 (모델, 에폭별 기록 표)를 돌려줍니다."""
    # 1. 모델 초기값과 미니배치 섞는 순서가 실행마다 같게 시드 고정
    torch.manual_seed(0)

    # 2. 학습용(섞기), 테스트용(섞지 않기) 미니배치
    train_loader = ...
    test_loader = ...

    # 3. 은닉층 두 개인 DNN과 손실함수
    model = ...
    loss_fn = nn.CrossEntropyLoss()

    # 4. optimizer_name에 맞는 옵티마이저
    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif optimizer_name == "momentum":
        optimizer = ...
    else:
        optimizer = ...

    # 5. 에폭마다 학습하고 학습 데이터, 테스트 데이터의 정확도를 기록
    rows = []
    for epoch in range(1, epochs + 1):
        train_loss = ...
        _, train_accuracy = ...
        _, test_accuracy = ...
        rows.append({"에폭": epoch, "학습 손실": train_loss, "학습 정확도": train_accuracy, "테스트 정확도": test_accuracy})

    # 6. (모델, 기록 표)
    return model, pd.DataFrame(rows)

In [ ]:
# [자가채점] 기록 표와 모델 모양을 보고, 1에폭 결과를 지문대로 만든 함수의 결과와 비교합니다.

# 숫자 입력 640행으로 2에폭 학습해 기록 표와 모델 모양을 확인
check_X_train = X_number_train[:640]
check_y_train = y_train[:640]
check_X_test = X_number_test[:64]
check_y_test = y_test[:64]
check_model, check_history = train_weekday_model(check_X_train, check_y_train, check_X_test, check_y_test, lr=0.1, epochs=2, optimizer_name="sgd")
assert isinstance(check_history, pd.DataFrame), "train_weekday_model은 (모델, 기록 표)를 돌려줍니다. `...`를 채웠는지 확인하세요."
assert list(check_history.columns) == ["에폭", "학습 손실", "학습 정확도", "테스트 정확도"], "기록 표의 열은 에폭, 학습 손실, 학습 정확도, 테스트 정확도 순서입니다."
assert list(check_history["에폭"]) == [1, 2], "epochs=2이면 기록 표는 에폭 1, 2 두 행입니다."
assert isinstance(check_model, nn.Sequential), "모델은 nn.Sequential로 만드세요."
assert check_model[0].in_features == 3, "첫 nn.Linear의 입력 크기는 X_train의 열 수입니다. 숫자 입력은 3칸입니다."
assert check_model[-1].out_features == 7, "마지막 nn.Linear의 출력 크기는 요일 수 7입니다."
assert len(check_model) == 5, "모델은 nn.Linear, nn.ReLU, nn.Linear, nn.ReLU, nn.Linear 다섯 층입니다. 입력 -> 128 -> ReLU -> 128 -> ReLU -> 7 구조인지 보세요."
assert check_model[2].in_features == 128 and check_model[2].out_features == 128, "가운데 nn.Linear는 128 -> 128입니다. 입력 -> 128 -> ReLU -> 128 -> ReLU -> 7 구조인지 보세요."
assert 0 <= check_history["테스트 정확도"].iloc[-1] <= 1, "정확도는 0~1 사이입니다. evaluate가 돌려주는 두 값 가운데 둘째 값을 쓰세요."

# 지문대로 만든 함수로 미리 구한 1에폭 결과(숫자 입력 640행, 시드 0): (학습 손실, 학습 정확도)
expected_first_epoch = {
    "sgd": (1.948993, 0.1671875),
    "momentum": (1.954110, 0.18125),
    "adam": (1.992766, 0.1734375),
}

# 옵티마이저마다 고칠 곳
fix_messages = {
    "sgd": "sgd 줄은 채워 두었으니 미니배치를 보세요. 학습용 DataLoader는 배치 크기 64, shuffle=True이고, 학습 손실은 train_one_epoch가 돌려준 값입니다.",
    "momentum": "\"momentum\"이면 torch.optim.SGD에 lr=lr과 momentum=0.9를 주세요. lr을 빼면 기본값 0.001로 학습합니다.",
    "adam": "\"adam\"이면 torch.optim.Adam에 lr=lr을 주세요. lr을 빼면 기본값 0.001로 학습합니다.",
}
for check_name, check_lr in [("sgd", 0.1), ("momentum", 0.1), ("adam", 0.01)]:
    # 같은 데이터로 1에폭 학습해 미리 구한 결과와 비교
    name_model, name_history = train_weekday_model(check_X_train, check_y_train, check_X_test, check_y_test, lr=check_lr, epochs=1, optimizer_name=check_name)
    expected_loss, expected_accuracy = expected_first_epoch[check_name]

    # 학습 손실이 다르면 미니배치나 옵티마이저 설정이 지문과 다릅니다
    student_loss = name_history["학습 손실"].iloc[0]
    assert abs(student_loss - expected_loss) < 1e-3, f"\"{check_name}\"(학습률 {check_lr})로 1에폭 학습한 손실이 기대값과 다릅니다(내 함수 {student_loss:.4f}, 기대값 {expected_loss:.4f}). {fix_messages[check_name]} 지문 조건: 배치 크기 64, 학습용 shuffle=True, 옵티마이저에 lr=lr, 모멘텀은 momentum=0.9."

    # 손실이 같은데 학습 정확도가 다르면 정확도를 구한 로더가 다릅니다
    student_accuracy = name_history["학습 정확도"].iloc[0]
    assert abs(student_accuracy - expected_accuracy) < 0.005, f"\"{check_name}\"의 학습 정확도가 기대값과 다릅니다(내 함수 {student_accuracy:.4f}, 기대값 {expected_accuracy:.4f}). 학습 정확도는 evaluate에 학습용 로더(train_loader)를 넘겨 구하세요."
print("1-2 확인 완료")

### 1-3. 숫자 입력으로 학습하기

**요구사항**: 숫자 입력(`X_number_train`, `X_number_test`)을 학습률 0.001, 20에폭, `"adam"`으로 학습해 모델은 **`number_model`**, 기록 표는 **`number_history`** 에 담고 기록 표를 보여 주세요.

**확인 기준**: 기록 표는 20행이고, 마지막 에폭에서 학습 손실은 1.8보다 크고(log 7 = 1.95 근처), 학습 정확도와 테스트 정확도는 0.25보다 작습니다(찍기 수준).

In [ ]:
# [작성] 숫자 입력으로 20에폭 학습하세요.
number_model, number_history = ...
display(number_history.round(3))

In [ ]:
# [자가채점] 숫자 입력의 학습 결과가 찍기 수준에 머무는지 확인합니다.
assert isinstance(number_history, pd.DataFrame) and len(number_history) == 20, "number_history는 20에폭의 기록 표(20행)입니다. `...`를 채웠는지 확인하세요."
assert number_model[0].in_features == 3, "숫자 입력이면 첫 nn.Linear의 입력 크기가 3입니다. X_number_train을 넘겼는지 보세요."

number_last = number_history.iloc[-1]
assert number_last["학습 손실"] > 1.8, f"마지막 학습 손실이 {number_last['학습 손실']:.3f}입니다. 숫자 입력이면 log 7 = 1.95 근처에 머뭅니다. 학습률 0.001, 20에폭, adam으로 학습했는지 보세요."
assert number_last["학습 정확도"] < 0.25, f"마지막 학습 정확도가 {number_last['학습 정확도']:.3f}입니다. 숫자 입력이면 찍기 수준(약 0.14)입니다."
assert 0.05 < number_last["테스트 정확도"] < 0.25, f"마지막 테스트 정확도가 {number_last['테스트 정확도']:.3f}입니다. 숫자 입력이면 찍기 수준(약 0.14)입니다. 테스트용 로더로 평가했는지 보세요."
print("1-3 확인 완료: 마지막 학습 손실", round(number_last["학습 손실"], 3), "| 테스트 정확도", round(number_last["테스트 정확도"], 3))

## 2. 학습률을 바꿔 보기

**배경**: 손실이 줄지 않으면 학습률부터 의심하는 것이 교안 01·02의 순서였습니다. 10배 간격 학습률 네 개로 다시 학습해, 손실이 log 7 근처에 머문 원인이 학습률인지 확인합니다.

**요구사항**
- 학습률 0.0001, 0.001, 0.01, 0.1마다 숫자 입력을 20에폭, `"adam"`으로 학습하세요.
- 학습률마다 `학습률`, `마지막 학습 손실`, `학습 정확도` 세 키의 dict를 만들어 표 **`lr_table`** 에 담으세요. 값은 기록 표의 마지막 에폭(맨 아래 행)에서 꺼냅니다.

**예시**: 학습률 0.001 행은 1-3과 같은 설정이므로 1-3 기록 표의 마지막 행과 값이 같습니다.

**확인 기준**: `lr_table`은 학습률이 작은 것부터 4행이고, 네 행 모두 마지막 학습 손실이 1.8보다 크고 학습 정확도가 0.25보다 작습니다.

In [ ]:
# [작성] 학습률 네 개로 숫자 입력 모델을 학습해 표로 비교하세요.
lr_rows = []
for lr in [0.0001, 0.001, 0.01, 0.1]:
    # 1. 숫자 입력, 20에폭, adam으로 학습
    _, history = ...

    # 2. 마지막 에폭(맨 아래 행)의 값으로 한 행
    last = history.iloc[-1]
    lr_rows.append(...)

lr_table = pd.DataFrame(lr_rows)
display(lr_table.round({"마지막 학습 손실": 4, "학습 정확도": 4}))

In [ ]:
# [자가채점] 표 모양과, 학습률을 바꿔도 찍기 수준에 머무는지 확인합니다.
assert isinstance(lr_table, pd.DataFrame) and len(lr_table) == 4, "lr_table은 학습률마다 한 행, 4행 표입니다. `...`를 채웠는지 확인하세요."
assert list(lr_table.columns) == ["학습률", "마지막 학습 손실", "학습 정확도"], "lr_table의 열은 학습률, 마지막 학습 손실, 학습 정확도 순서입니다. 설정을 비교하는 표라 테스트 정확도는 넣지 않습니다."
assert list(lr_table["학습률"]) == [0.0001, 0.001, 0.01, 0.1], "행은 학습률 0.0001, 0.001, 0.01, 0.1 순서입니다."

row_001 = lr_table.iloc[1]
assert abs(row_001["마지막 학습 손실"] - number_history.iloc[-1]["학습 손실"]) < 1e-4, "학습률 0.001 행은 1-3과 같은 설정이라 값도 같아야 합니다. 숫자 입력, 20에폭, adam으로 학습했는지 보세요."
assert (lr_table["마지막 학습 손실"] > 1.8).all(), "숫자 입력은 학습률을 바꿔도 손실이 log 7 = 1.95 근처에 머뭅니다. 입력이 X_number_train인지 보세요."
assert (lr_table["학습 정확도"] < 0.25).all(), "숫자 입력은 학습률을 바꿔도 학습 정확도가 찍기 수준(약 0.14)입니다. 기록 표의 학습 정확도 열을 꺼냈는지 보세요."
print("2 확인 완료")

**결과 읽기**: 학습률을 1,000배 범위(0.0001~0.1)로 바꿔도 마지막 학습 손실은 모두 1.95 근처, 학습 정확도는 0.14~0.15로 찍기 수준(약 0.14)입니다. 학습 데이터조차 맞히지 못하므로 손실이 줄지 않은 원인은 학습률이 아닙니다.

## 3. 원-핫 입력 만들기

**배경**: 학습률이 원인이 아니라면 입력을 바꿔 봅니다. 회귀분석 단원에서 `pd.get_dummies`로 범주형 변수를 0과 1로 된 열로 펼쳤습니다. 이것이 원-핫 인코딩(값마다 칸을 하나씩 만들고, 그 값의 칸만 1, 나머지 칸은 0으로 나타내는 방법)입니다. 이번에는 `연`, `월`, `일`에 씁니다. 회귀에서는 기준 범주 하나를 뺐지만 여기서는 값마다 칸을 모두 만듭니다.

<img src="images/41_date_onehot.png" width="1050" alt="날짜 2024년 10월 20일을 숫자 3개(2024, 10, 20)로 나타낸 입력과, 연 30칸, 월 12칸, 일 31칸 중 해당 칸만 1인 원-핫 73칸으로 나타낸 입력을 나란히 비교한 그림">

**요구사항**
- `요일`을 뺀 `연`, `월`, `일` 세 열을 `pd.get_dummies`로 펼친 표 **`onehot_table`** 을 만드세요. 열은 연 30칸(2000~2029), 월 12칸(1~12), 일 31칸(1~31), 모두 73칸입니다.
- **10,958행 전체를 먼저 펼친 뒤** 나눕니다. 학습 행과 테스트 행을 따로 펼치면 6절처럼 연도로 나눌 때 학습 쪽은 연 25칸, 테스트 쪽은 연 5칸이 되어 모양이 맞지 않습니다.
- `onehot_table`을 float32 텐서 **`X_onehot`** 으로 바꾸고, 학습 행만 고른 **`X_onehot_train`**, 테스트 행만 고른 **`X_onehot_test`** 를 만드세요.

**예시**: 첫 행(2000년 1월 1일)은 `연_2000`, `월_1`, `일_1` 세 칸만 1이고 나머지 70칸은 0입니다.

**확인 기준**: `X_onehot`은 `(10958, 73)` 모양의 float32 텐서이고, 행마다 1이 세 개(연, 월, 일 하나씩)이며 1인 자리가 그 행의 날짜와 맞습니다.

<details><summary>힌트</summary>

```text
접근방법:
- pd.get_dummies 는 글자 열만 저절로 펼친다. 연, 월, 일은 숫자 열이라 펼칠 열을 columns 로 직접 알려 줘야 한다.

세부구현:
1. dates 에서 연, 월, 일 세 열만 고른 표를 pd.get_dummies 에 넘긴다.
   columns 에 연, 월, 일 목록을, dtype 에 "float32" 를 준다.
   (columns 를 빼면 숫자 열이 그대로 남아 3칸, dtype 을 빼면 0, 1 대신 True, False 가 나온다)
2. to_numpy 로 바꿔 torch.tensor 에 넘기고 dtype 에 torch.float32 를 준다.
3. X_onehot 을 train_index, test_index 로 인덱싱한다.
```

</details>

In [ ]:
# [작성] 연, 월, 일을 원-핫으로 펼친 입력을 만드세요.

# 1. 연, 월, 일 세 열을 원-핫 표로 펼치기(요일은 정답이라 넣지 않음)
onehot_table = ...
print("원-핫 표 모양:", onehot_table.shape)

# 첫 행에서 1인 칸의 이름
first_row = onehot_table.iloc[0]
print("첫 행에서 1인 칸:", list(onehot_table.columns[first_row == 1]))

# 2. float32 텐서
X_onehot = ...

# 3. 학습 입력과 테스트 입력
X_onehot_train = ...
X_onehot_test = ...

In [ ]:
# [자가채점] 원-핫 입력의 모양과, 1인 자리가 날짜와 맞는지 확인합니다.
assert isinstance(X_onehot, torch.Tensor) and X_onehot.dim() == 2 and len(X_onehot) == len(dates), "X_onehot은 날짜마다 한 행(10,958행)인 2차원 텐서입니다. `...`를 채웠는지, 전체 행을 펼쳤는지 확인하세요."
assert X_onehot.shape[1] == 73, f"X_onehot의 열은 연 30 + 월 12 + 일 31 = 73칸입니다. 지금은 {X_onehot.shape[1]}칸입니다. 3칸이면 columns를 빠뜨린 것이고, 74칸이나 80칸이면 요일 열이 남은 것입니다."
assert X_onehot.dtype == torch.float32, "X_onehot은 float32 텐서로 만드세요. torch.tensor에 dtype=torch.float32를 주면 됩니다."
assert torch.equal(X_onehot.sum(dim=1), torch.full((len(dates),), 3.0)), "원-핫 입력은 행마다 1이 정확히 세 개(연, 월, 일 하나씩)이고 나머지는 0입니다."

years = torch.tensor(dates["연"].to_numpy())
months = torch.tensor(dates["월"].to_numpy())
days = torch.tensor(dates["일"].to_numpy())
assert torch.equal(X_onehot[:, :30].argmax(dim=1), years - 2000), "앞 30칸은 연 2000~2029입니다. 열 순서가 연, 월, 일인지 보세요."
assert torch.equal(X_onehot[:, 30:42].argmax(dim=1), months - 1), "가운데 12칸은 월 1, 2, ..., 12 순서입니다. 숫자를 글자로 바꿔 펼치면 1, 10, 11, 12, 2 순서가 되니, 숫자 열 그대로 columns에 지정해 펼치세요."
assert torch.equal(X_onehot[:, 42:].argmax(dim=1), days - 1), "마지막 31칸은 일 1, 2, ..., 31 순서입니다. 숫자를 글자로 바꿔 펼치면 순서가 바뀝니다."
assert torch.equal(X_onehot_train, X_onehot[train_index]), "X_onehot_train은 X_onehot에서 train_index 행을 고른 것입니다."
assert torch.equal(X_onehot_test, X_onehot[test_index]), "X_onehot_test는 X_onehot에서 test_index 행을 고른 것입니다."
print("3 확인 완료")

## 4. 원-핫 입력으로 학습하기

**배경**: 1-3과 모델 구조, 학습률, 에폭 수, 옵티마이저를 모두 같게 두고 입력만 원-핫으로 바꿉니다. 입력 표현 하나가 결과를 얼마나 바꾸는지 확인합니다.

**요구사항**: 원-핫 입력(`X_onehot_train`, `X_onehot_test`)을 학습률 0.001, 20에폭, `"adam"`으로 학습해 모델은 **`onehot_model`**, 기록 표는 **`onehot_history`** 에 담고 기록 표를 보여 주세요.

**확인 기준**: 기록 표는 20행이고, 마지막 에폭의 학습 정확도와 테스트 정확도가 모두 0.95를 넘습니다.

In [ ]:
# [작성] 원-핫 입력으로 20에폭 학습하세요.
onehot_model, onehot_history = ...
display(onehot_history.round(3))

In [ ]:
# [자가채점] 원-핫 입력으로 학습한 결과를 확인합니다.
assert isinstance(onehot_history, pd.DataFrame) and len(onehot_history) == 20, "onehot_history는 20에폭의 기록 표(20행)입니다. `...`를 채웠는지 확인하세요."
assert onehot_model[0].in_features == 73, "원-핫 입력이면 첫 nn.Linear의 입력 크기가 73입니다. X_onehot_train을 넘겼는지 보세요."

onehot_last = onehot_history.iloc[-1]
assert onehot_last["학습 정확도"] > 0.95, f"마지막 학습 정확도가 {onehot_last['학습 정확도']:.3f}입니다. 0.95를 넘어야 합니다. 함수 안에서 train_one_epoch로 실제로 학습하는지 보세요. 학습용 DataLoader에 shuffle=True를 줬는지도 보세요."
assert onehot_last["테스트 정확도"] > 0.95, f"마지막 테스트 정확도가 {onehot_last['테스트 정확도']:.3f}입니다. 0.95를 넘어야 합니다. 테스트 입력이 X_onehot_test인지 보세요. 학습용 DataLoader에 shuffle=True를 줬는지도 보세요."
print("4 확인 완료: 마지막 테스트 정확도", round(onehot_last["테스트 정확도"], 3))

아래 셀은 실행만 하세요. 두 입력의 에폭별 학습 손실과 테스트 정확도를 나란히 그립니다. 회색 점선은 찍기 수준(손실 log 7, 정확도 1/7)입니다.

In [ ]:
import math

# 숫자 입력과 원-핫 입력의 학습 곡선
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for history, label in [(number_history, "숫자 입력(3칸)"), (onehot_history, "원-핫 입력(73칸)")]:
    axes[0].plot(history["에폭"], history["학습 손실"], marker="o", label=label)
    axes[1].plot(history["에폭"], history["테스트 정확도"], marker="o", label=label)

# 찍기 수준 기준선
axes[0].axhline(math.log(7), color="gray", linestyle="--", label="log 7")
axes[1].axhline(1 / 7, color="gray", linestyle="--", label="1/7")

axes[0].set_title("에폭별 학습 손실", fontsize=14)
axes[1].set_title("에폭별 테스트 정확도", fontsize=14)
for ax in axes:
    ax.set_xlabel("에폭", fontsize=12)
    ax.set_xticks(range(2, 21, 2))
    ax.tick_params(labelsize=11)
    ax.legend(loc="center right", fontsize=11)  # 두 선이 위아래로 갈라지는 가운데 오른쪽 빈자리에 둡니다
plt.tight_layout()
plt.show()

**결과 읽기**
- 숫자 입력(파란 선)은 20에폭 내내 손실이 log 7 선(1.95), 테스트 정확도(0.11)가 1/7 선(0.14) 근처에 머뭅니다. 요일은 `일`이 하나 늘 때마다 바뀌고 7일마다 처음으로 돌아와 값의 크기와 상관이 없어, 크기를 그대로 받는 입력으로는 배우지 못했습니다.
- 원-핫 입력(주황 선)은 값마다 자기 칸과 가중치가 있어 크기 순서를 가정하지 않습니다. 처음 4에폭은 기준선 근처에 머물다가 5에폭부터 손실이 줄어 20에폭에는 테스트 정확도 0.99입니다. 처음 손실이 거의 줄지 않는 구간을 5절에서 다룹니다.

## 5. 옵티마이저와 학습률 비교하기

**배경**: 4절 원-핫 입력도 처음 4에폭은 손실이 log 7 근처에 머물렀습니다. 같은 5에폭 안에 어느 설정이 이 구간을 빨리 벗어나 학습 손실을 줄이는지 비교합니다. 20에폭이면 학습이 되는 Adam 설정들은 학습 데이터를 거의 다 맞혀 차이가 보이지 않으므로 5에폭에서 비교합니다.

**요구사항**
- 아래 여섯 설정마다 원-핫 입력을 5에폭 학습하세요(표 순서 그대로).

| 옵티마이저 이름 | 학습률 |
|---|---|
| `"sgd"` | 0.1 |
| `"momentum"` | 0.1 |
| `"adam"` | 0.001 |
| `"adam"` | 0.003 |
| `"adam"` | 0.01 |
| `"adam"` | 0.03 |

- 설정마다 `옵티마이저`, `학습률`, `마지막 학습 손실` 세 키의 dict를 만들어 표 **`optimizer_table`** 에 담으세요. `옵티마이저` 값은 위 표의 옵티마이저 이름입니다.
- 골격에 적힌 줄이 설정마다 기록 표를 **`optimizer_histories`** 에 모읍니다(아래 그래프가 씁니다). 이 줄은 그대로 둡니다.

**예시**: 첫 행은 `{"옵티마이저": "sgd", "학습률": 0.1, "마지막 학습 손실": 1.95}`처럼 생겼습니다(손실 값은 예).

**확인 기준**: `optimizer_table`은 위 표 순서의 6행입니다. SGD와 모멘텀은 5에폭 뒤에도 학습 손실이 1.9보다 크고, Adam 0.003은 1.0보다 작습니다.

In [ ]:
# [작성] 여섯 설정으로 원-핫 입력을 5에폭씩 학습해 표로 비교하세요.
optimizer_rows = []
optimizer_histories = {}  # 설정 이름 -> 기록 표(아래 그래프가 씁니다)
for optimizer_name, lr in [("sgd", 0.1), ("momentum", 0.1), ("adam", 0.001), ("adam", 0.003), ("adam", 0.01), ("adam", 0.03)]:
    # 1. 원-핫 입력, 5에폭으로 학습
    _, history = ...
    optimizer_histories[f"{optimizer_name} {lr}"] = history

    # 2. 마지막 에폭의 학습 손실로 한 행
    last = history.iloc[-1]
    optimizer_rows.append(...)

optimizer_table = pd.DataFrame(optimizer_rows)
display(optimizer_table.round(3))

In [ ]:
# [자가채점] 표 모양과 비교 결과의 방향을 확인합니다.
assert isinstance(optimizer_table, pd.DataFrame) and len(optimizer_table) == 6, "optimizer_table은 설정마다 한 행, 6행 표입니다. `...`를 채웠는지 확인하세요."
assert list(optimizer_table.columns) == ["옵티마이저", "학습률", "마지막 학습 손실"], "optimizer_table의 열은 옵티마이저, 학습률, 마지막 학습 손실 순서입니다."
assert list(optimizer_table["옵티마이저"]) == ["sgd", "momentum", "adam", "adam", "adam", "adam"], "행은 지문 표의 순서(sgd, momentum, adam 네 개)입니다."
assert list(optimizer_table["학습률"]) == [0.1, 0.1, 0.001, 0.003, 0.01, 0.03], "학습률은 지문 표의 순서(0.1, 0.1, 0.001, 0.003, 0.01, 0.03)입니다."
assert len(optimizer_histories["adam 0.01"]) == 5, "에폭 수는 5입니다."

losses = list(optimizer_table["마지막 학습 손실"])
assert losses[0] > 1.9 and losses[1] > 1.9, f"SGD, 모멘텀(0.1)의 마지막 학습 손실이 {losses[0]:.3f}, {losses[1]:.3f}입니다. 이 설정이면 5에폭 뒤에도 log 7 근처입니다. 원-핫 입력과 optimizer_name을 바르게 넘겼는지 보세요."
assert losses[3] < 1.0, f"Adam 0.003의 마지막 학습 손실이 {losses[3]:.3f}입니다. 1.0보다 작아야 합니다. 원-핫 입력으로 학습했는지, adam에 lr=lr을 줬는지 보세요."
print("5 확인 완료")

아래 셀은 실행만 하세요. 설정마다 에폭별 학습 손실을 그립니다. 선이 빨리 내려갈수록 같은 에폭 안에 더 많이 배운 설정입니다.

In [ ]:
# 설정마다 에폭별 학습 손실
fig, ax = plt.subplots(figsize=(9, 4.5))
# log 7 근처에서 겹치는 세 설정은 선 모양, 속이 빈 마커, 마커 크기를 달리해 겹쳐도 셋 다 보이게 합니다
line_styles = {"sgd 0.1": ("--", "s", 14), "momentum 0.1": (":", "^", 10), "adam 0.03": ("-.", "o", 6)}
for setting, history in optimizer_histories.items():
    if setting in line_styles:
        linestyle, marker, size = line_styles[setting]
        ax.plot(history["에폭"], history["학습 손실"], linestyle=linestyle, marker=marker, markersize=size,
                markerfacecolor="none", markeredgewidth=1.5, label=setting)
    else:
        ax.plot(history["에폭"], history["학습 손실"], marker="o", label=setting)
ax.axhline(math.log(7), color="gray", linestyle="--", linewidth=1, zorder=0, label="log 7")
ax.set_title("옵티마이저와 학습률별 학습 손실(원-핫 입력, 5에폭)")
ax.set_xlabel("에폭")
ax.set_ylabel("학습 손실")
ax.set_xticks(range(1, 6))
ax.legend(loc="center left", bbox_to_anchor=(1, 0.5))  # 범례가 선을 가리지 않게 그래프 오른쪽 밖에 둡니다
plt.tight_layout()
plt.show()

**결과 읽기**
- `sgd 0.1`, `momentum 0.1`, `adam 0.03` 세 선은 log 7 점선 위에 겹쳐 있어, 속이 빈 마커(큰 네모, 세모, 작은 동그라미)가 한자리에 포개져 보입니다. 5에폭 뒤 학습 손실이 모두 1.95 근처로 거의 줄지 않았습니다. Adam도 학습률이 너무 크면(0.03) 줄지 않습니다.
- Adam 0.001은 1.8 근처에 머물고, 0.003과 0.01은 1.0 아래로 내려갑니다. 0.003과 0.01 가운데 어느 쪽이 더 낮은지는 시드(초기 파라미터와 섞는 순서)를 바꾸면 뒤집히고, 시드에 따라서는 0.01이 5에폭 안에 1.0 아래로 내려가지 못하기도 합니다. 그래서 5-1에서는 마지막 학습 손실의 순위가 아니라 교안 01의 규칙으로 고릅니다.

### 5-1. 설정 고르기

**배경**: 설정은 학습 손실로 고릅니다. 테스트 데이터는 마지막 확인에만 씁니다. 테스트 정확도를 보고 설정을 고르면, 그 테스트 정확도는 처음 보는 데이터의 성능보다 좋게 나옵니다(이 문제를 푸는 방법은 다음 시간에 배웁니다).

- 교안 01에서 정한 순서(10배 간격으로 넓게 시도 -> 발산 경계 찾기 -> 정한 횟수 안에 수렴한 후보 가운데 발산 경계에서 가장 먼 값)로 고릅니다.
- **수렴한 후보**는 5에폭 뒤 마지막 학습 손실이 **1.0 아래**인 설정으로 정합니다. 찍기 수준 손실이 log 7(1.95)이므로 1.0 아래면 정체 구간을 벗어나 손실이 내려온 것입니다. SGD 0.1과 모멘텀 0.1은 1.0 아래로 내려가지 않아 후보에 들지 않습니다.
- Adam은 0.03에서 손실이 줄지 않았으므로 발산 경계는 0.03 근처입니다. 수렴한 Adam 후보는 모두 이 경계보다 작으므로, 경계에서 가장 먼 값은 **학습률이 가장 작은 후보**입니다. 경계에 가까운 값은 시드나 데이터가 조금만 바뀌어도 정체 구간을 못 벗어날 수 있습니다.

**요구사항**: `optimizer_table`에서 마지막 학습 손실이 1.0 아래인 행만 남긴 표를 **`converged_table`**(DataFrame, 열은 `optimizer_table`과 같음)에 담고, 그 가운데 학습률이 가장 작은 행을 **`best_row`**(Series)에 담아 고른 옵티마이저와 학습률을 출력하세요.

**확인 기준**: `converged_table`에 마지막 학습 손실이 1.0 아래인 행만 있고, `best_row`가 그 가운데 학습률이 가장 작은 행입니다.

<details><summary>힌트</summary>

```text
접근방법:
- 표 -> 수렴한 행만 남기기 -> 남은 행 가운데 학습률이 가장 작은 행

세부구현:
1. 행을 조건으로 남길 때는 표[조건] 형태로 쓴다. 조건은 마지막 학습 손실 열 < 1.0 이다.
2. 남은 표를 학습률 열로 오름차순 정렬한다(sort_values).
3. 정렬한 표의 맨 위 행을 iloc 으로 꺼낸다.
```

</details>

In [ ]:
# [작성] 수렴한 후보 가운데 발산 경계에서 가장 먼 설정을 고르세요.

# 1. 5에폭 뒤 마지막 학습 손실이 1.0 아래인 설정만 남기기
converged_table = ...
display(converged_table.round(3))

# 2. 남은 설정 가운데 학습률이 가장 작은 행
best_row = ...
print("고른 설정:", best_row["옵티마이저"], best_row["학습률"], "| 마지막 학습 손실:", round(best_row["마지막 학습 손실"], 4))

In [ ]:
# [자가채점] 수렴한 후보와 고른 설정을 확인합니다.

# 수렴한 후보: 마지막 학습 손실이 1.0 아래인 설정
expected_table = optimizer_table[optimizer_table["마지막 학습 손실"] < 1.0]
expected_settings = list(zip(expected_table["옵티마이저"], expected_table["학습률"]))
assert isinstance(converged_table, pd.DataFrame), "converged_table은 optimizer_table에서 행을 골라 남긴 DataFrame입니다. `...`를 채웠는지 확인하세요."
assert list(zip(converged_table["옵티마이저"], converged_table["학습률"])) == expected_settings, f"converged_table에는 마지막 학습 손실이 1.0 아래인 설정 {expected_settings}만 남깁니다. 조건이 마지막 학습 손실 열 < 1.0인지 보세요."

# best_row: 수렴한 후보 가운데 발산 경계에서 가장 먼 값, 곧 학습률이 가장 작은 행
assert isinstance(best_row, pd.Series), "best_row는 converged_table의 한 행입니다. 정렬한 표에서 iloc으로 꺼내세요."
assert best_row["마지막 학습 손실"] < 1.0, f"고른 설정의 마지막 학습 손실이 {best_row['마지막 학습 손실']:.3f}입니다. optimizer_table이 아니라 수렴한 후보만 남긴 converged_table에서 고르세요."
assert best_row["학습률"] == expected_table["학습률"].min(), f"고른 학습률 {best_row['학습률']}이 수렴한 후보 가운데 가장 작은 학습률({expected_table['학습률'].min()})이 아닙니다. 마지막 학습 손실이 가장 낮은 행이 아니라 발산 경계(0.03)에서 가장 먼 행을 고릅니다. 학습률 열로 오름차순 정렬했는지 보세요."
print("5-1 확인 완료")

## 6. 처음 보는 연도 맞히기

**배경**: 4절의 테스트 행은 무작위로 뽑았으므로 같은 해의 다른 날짜가 학습 데이터에 들어 있었습니다. 실제로 요일을 맞혀야 하는 날짜는 앞으로 올 날짜입니다. 2024년까지만 학습하고 2025~2029년을 맞혀, 학습 데이터에 없던 연도에도 원-핫 입력이 통하는지 확인합니다.

**요구사항**
- `연`이 2024 이하인 행 번호를 정수 텐서 **`past_index`** 에, 2025 이상인 행 번호를 **`future_index`** 에 담으세요.
- `X_onehot`과 `y_all`에서 `past_index` 행으로 학습하고 `future_index` 행으로 테스트하세요. 학습률 0.001, 20에폭, `"adam"`입니다. 모델은 **`future_model`**, 기록 표는 **`future_history`** 에 담고 기록 표를 보여 주세요.

**예시**: 2024년 12월 31일은 `past_index`, 2025년 1월 1일은 `future_index`에 들어갑니다. `past_index`는 9,132개, `future_index`는 1,826개입니다.

**확인 기준**: 학습 행에 2025년 이후가 하나도 없고, 마지막 에폭의 학습 정확도는 0.95를 넘지만 테스트 정확도는 0.3보다 작습니다.

<details><summary>힌트</summary>

```text
세부구현:
1. dates 의 연 열로 2024 이하 조건을 만들고, dates.index 를 그 조건으로 걸러 행 번호를 얻는다.
2. 1의 행 번호를 torch.tensor 로 바꾼다. 2025 이상도 같은 방법으로 만든다.
3. X_onehot 과 y_all 을 past_index 로 인덱싱하면 학습 입력과 정답, future_index 로 인덱싱하면 테스트 입력과 정답이다.
4. train_weekday_model 에 3의 네 텐서를 넘긴다.
```

</details>

In [ ]:
# [작성] 2024년까지로 학습하고 2025~2029년으로 테스트하세요.

# 1. 2024년까지인 행 번호와 2025년부터인 행 번호
past_index = ...
future_index = ...
print("학습 행:", len(past_index), "| 테스트 행:", len(future_index))

# 2. 과거 행으로 학습하고 미래 행으로 테스트
future_model, future_history = ...
display(future_history.round(3))

In [ ]:
# [자가채점] 연도로 나눈 행 번호와, 처음 보는 연도의 결과를 확인합니다.
assert isinstance(past_index, torch.Tensor) and isinstance(future_index, torch.Tensor), "past_index, future_index는 행 번호를 담은 정수 텐서입니다. `...`를 채웠는지 확인하세요."
years = torch.tensor(dates["연"].to_numpy())
assert len(past_index) + len(future_index) == len(dates), "past_index와 future_index를 합치면 모든 행(10,958개)입니다."
assert years[past_index].max().item() == 2024, "past_index에는 2024년까지만 들어갑니다. 2025년 이후 행이 학습에 섞이면 처음 보는 연도가 아니게 됩니다."
assert years[future_index].min().item() == 2025, "future_index에는 2025년부터 들어갑니다."
assert isinstance(future_history, pd.DataFrame) and len(future_history) == 20, "future_history는 20에폭의 기록 표(20행)입니다."

future_last = future_history.iloc[-1]
assert future_last["학습 정확도"] > 0.95, f"마지막 학습 정확도가 {future_last['학습 정확도']:.3f}입니다. 학습한 연도는 0.95를 넘게 맞힙니다. 학습 입력이 X_onehot[past_index]인지, 학습 정확도를 학습용 로더로 구했는지 보세요. 학습용 DataLoader에 shuffle=True를 줬는지도 보세요."
assert future_last["테스트 정확도"] < 0.3, f"마지막 테스트 정확도가 {future_last['테스트 정확도']:.3f}입니다. 처음 보는 연도는 찍기 수준이어야 합니다. 테스트에 future_index 행을 넘겼는지, 학습에 2025년 이후 행이 섞이지 않았는지 보세요. 학습용 DataLoader에 shuffle=True를 줬는지도 보세요."
print("6 확인 완료: 학습 정확도", round(future_last["학습 정확도"], 3), "| 테스트 정확도", round(future_last["테스트 정확도"], 3))

아래 셀은 실행만 하세요. 4절(테스트 날짜의 연도를 학습에서 봄)과 6절(테스트 연도를 학습에서 못 봄)의 마지막 에폭 정확도를 막대로 비교합니다.

In [ ]:
# 무작위로 나눈 실험(4절)과 연도로 나눈 실험(6절)의 마지막 에폭 정확도
split_names = ["무작위로 나눔(4절)", "연도로 나눔(6절)"]
train_accuracies = [onehot_history["학습 정확도"].iloc[-1], future_history["학습 정확도"].iloc[-1]]
test_accuracies = [onehot_history["테스트 정확도"].iloc[-1], future_history["테스트 정확도"].iloc[-1]]

# 학습 정확도와 테스트 정확도 막대를 나란히 놓으려고 x 위치를 0.2씩 비킵니다
fig, ax = plt.subplots(figsize=(7, 4.5))
positions = range(len(split_names))
ax.bar([p - 0.2 for p in positions], train_accuracies, width=0.4, label="학습 정확도")
ax.bar([p + 0.2 for p in positions], test_accuracies, width=0.4, label="테스트 정확도")
ax.axhline(1 / 7, color="gray", linestyle="--", label="1/7")

# 막대 위에 값 적기(1/7 점선과 겹치지 않게 막대 끝에서 0.05 위)
for p in positions:
    ax.text(p - 0.2, train_accuracies[p] + 0.05, f"{train_accuracies[p]:.2f}", ha="center")
    ax.text(p + 0.2, test_accuracies[p] + 0.05, f"{test_accuracies[p]:.2f}", ha="center")

ax.set_xticks(list(positions))
ax.set_xticklabels(split_names)
ax.set_ylim(0, 1.15)
ax.set_title("원-핫 입력의 마지막 에폭 정확도")
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

**결과 읽기**
- 무작위로 나눈 4절은 테스트 정확도 0.99인데, 연도로 나눈 6절은 학습 정확도 1.00에 테스트 정확도 0.11로 찍기 수준입니다.
- 2025~2029년 칸은 학습 데이터에서 늘 0이라 그 칸에 곱해지는 가중치의 기울기도 늘 0입니다. 그 가중치는 한 번도 갱신되지 않아, 모델은 처음 보는 연도가 요일을 며칠 미는지 모릅니다. 원-핫 입력은 학습에서 본 값만 배웁니다.